# 06 Monitoring

Purpose: review the model monitoring outputs after batch inference. This notebook is aligned with the Airflow monitoring DAG and focuses on monthly model performance, default rates, prediction score distributions, and model metadata.

## Rationale

The assignment pipeline now keeps monitoring focused on performance metrics and prediction outputs. Distribution drift tables and charts are outside the current submission scope.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession, functions as F

PROJECT_DIR = Path.cwd()
DATAMART_DIR = PROJECT_DIR / "datamart"
MODEL_BANK_DIR = PROJECT_DIR / "outputs" / "model_bank"

spark = (
    SparkSession.builder
    .appName("assignment2_monitoring_review")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

spark

In [ ]:
def spark_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/")


def display_pdf(df, limit=20):
    display(df.limit(limit).toPandas())

## Optional Regeneration

Leave `RUN_MONITORING = False` for review-only notebook use. Set it to `True` only when you want the notebook to regenerate the same monitoring table and figures as the Airflow monitoring DAG.

In [ ]:
RUN_MONITORING = False

if RUN_MONITORING:
    from utils.model_monitoring import run_model_monitoring

    monitoring_pd = run_model_monitoring()
    print(monitoring_pd.shape)

## Load Monitoring Outputs

In [ ]:
monitoring = spark.read.parquet(spark_path(DATAMART_DIR / "gold" / "model_monitoring"))
predictions = spark.read.parquet(spark_path(DATAMART_DIR / "gold" / "model_predictions"))

print(f"Monitoring rows: {monitoring.count():,}")
print(f"Prediction rows: {predictions.count():,}")
monitoring.printSchema()

## Monthly Performance

In [ ]:
monthly_perf = monitoring.select(
    "monitoring_month",
    "model_version",
    "row_count",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive",
).orderBy("monitoring_month")

display_pdf(monthly_perf, 30)

In [ ]:
perf_pd = monthly_perf.toPandas()

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=perf_pd, x="monitoring_month", y="roc_auc", marker="o", label="ROC AUC", ax=ax)
sns.lineplot(data=perf_pd, x="monitoring_month", y="average_precision", marker="o", label="Average precision", ax=ax)
sns.lineplot(data=perf_pd, x="monitoring_month", y="recall", marker="o", label="Recall", ax=ax)
ax.set_title("Monthly model performance")
ax.set_xlabel("Monitoring month")
ax.set_ylabel("Metric")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## Default Rate Monitoring

In [ ]:
default_rate = monitoring.select(
    "monitoring_month",
    "row_count",
    "actual_default_rate",
    "predicted_default_rate",
).orderBy("monitoring_month")

display_pdf(default_rate, 30)

In [ ]:
rate_pd = default_rate.toPandas().melt(
    id_vars=["monitoring_month", "row_count"],
    value_vars=["actual_default_rate", "predicted_default_rate"],
    var_name="rate_type",
    value_name="rate",
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=rate_pd, x="monitoring_month", y="rate", hue="rate_type", marker="o", ax=ax)
ax.set_title("Actual vs predicted default rate")
ax.set_xlabel("Monitoring month")
ax.set_ylabel("Rate")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Prediction Score Review

In [ ]:
score_summary = (
    predictions
    .withColumn("monitoring_month", F.date_format("feature_snapshot_date", "yyyy-MM"))
    .groupBy("monitoring_month", "model_version", "dataset_split")
    .agg(
        F.count("*").alias("row_count"),
        F.avg("prediction_score").alias("mean_prediction_score"),
        F.min("prediction_score").alias("min_prediction_score"),
        F.expr("percentile_approx(prediction_score, 0.5)").alias("median_prediction_score"),
        F.max("prediction_score").alias("max_prediction_score"),
        F.avg("prediction_label").alias("predicted_default_rate"),
    )
    .orderBy("monitoring_month", "dataset_split")
)

display_pdf(score_summary, 50)

In [ ]:
prediction_months = [
    row.monitoring_month
    for row in predictions.withColumn("monitoring_month", F.date_format("feature_snapshot_date", "yyyy-MM"))
    .select("monitoring_month")
    .distinct()
    .orderBy("monitoring_month")
    .collect()
]
selected_months = prediction_months[:2] + prediction_months[-4:] if len(prediction_months) > 6 else prediction_months

score_pd = (
    predictions
    .withColumn("monitoring_month", F.date_format("feature_snapshot_date", "yyyy-MM"))
    .where(F.col("monitoring_month").isin(selected_months))
    .select("monitoring_month", "prediction_score")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(
    data=score_pd,
    x="prediction_score",
    hue="monitoring_month",
    bins=20,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax,
)
ax.set_title("Prediction score distribution by selected months")
ax.set_xlabel("Prediction score")
plt.tight_layout()
plt.show()

## Model Metadata

In [ ]:
import json

with open(MODEL_BANK_DIR / "champion_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

metadata_summary = {
    "model_name": metadata.get("model_name"),
    "model_version": metadata.get("model_version"),
    "mlflow_registered_model_name": metadata.get("mlflow_registered_model_name"),
    "mlflow_registered_model_version": metadata.get("mlflow_registered_model_version"),
    "training_window": f"{metadata.get('training_start_date')} to {metadata.get('training_end_date')}",
    "test_window": f"{metadata.get('test_start_date')} to {metadata.get('test_end_date')}",
    "oot_window": f"{metadata.get('oot_start_date')} to {metadata.get('oot_end_date')}",
    "feature_count": len(metadata.get("features", [])),
}

pd.DataFrame([metadata_summary])

## Monitoring Notes

- Use monthly ROC AUC, average precision, recall, F1, and default-rate comparison to identify model performance degradation.
- Use prediction score distribution as an operational sanity check for unusual score concentration or scoring output changes.
- Retrain or investigate when performance degrades materially for consecutive monitoring months or when upstream data/business policy changes.